In [10]:
import pandas as pd
import os 
from pathlib import Path
import numpy as np 
import pickle
import random
import re

In [234]:
def parse_response_text(response_text, to_string=True):
    
    clean_text = re.sub(r'[ \t\n\r\f\v\-_.,，、。！？!?:：;；]', '', response_text)

    # Total relevant characters
    total_chars = len(clean_text)

    # Count Chinese characters
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', clean_text))

    # Compute ratio
    chinese_ratio = chinese_chars / total_chars if total_chars > 0 else 0
    # print(f"Chinese ratio: {chinese_ratio:.2f} ({chinese_chars}/{total_chars})")
    # Set flag if ratio > 70%
    if chinese_ratio > 0.7:
        # print("Response text is likely in Chinese.")
        chinese_flag = True
    
    # if '，' in response_text or '、' in response_text:
        chinese_flag = True
        # remove anything before ：
        # print(f"response_text: {response_text}")
        response_text = re.sub(r'.*：', '', response_text)
        response_text = re.sub(',', '，', response_text)  # replace commas with Chinese commas
        response_text = re.sub('、', '，', response_text)  # replace Chinese commas with Chinese commas
        
        # if there is no '，', change ' ' to '，'
        if '，' not in response_text and response_text.count('\n') < 3:
            response_text = re.sub(' ', '，', response_text)
        
        # print(f"response_text after removing prefix: {response_text}")
    else:
        chinese_flag = False
    
    if response_text.isascii():
        # use , to split the response text
        response_words = [x.strip().lower() for x in response_text.split(',')]
    else:
        # use ， to split the response text
        response_words = [x.strip().lower() for x in response_text.split('，')]
    response_words_raw = response_words.copy()
        
 
    re_parsing_list = [
        r'[-\*]\s+(.+?)(?=\n-|\n*$)',
        r'^\d+\.\s+(.+)$',
    ]
    
    response_words_re = None 
    
    for pattern in re_parsing_list:
        temp_response_words_re = re.findall(pattern, response_text, flags=re.MULTILINE) # just handle the llama vanilla case 
        if response_words_re is None or len(temp_response_words_re) > len(response_words_re):
            response_words_re = temp_response_words_re
            
    if len(response_words_re) < 2:
        # print(f"doing additional regex parsing for response_text: {response_text}")
        response_words_re = re.findall(r'[A-Za-z0-9_-]+(?:[ \t]+[A-Za-z0-9_-]+){0,2}[\r\n]', response_text, re.MULTILINE)
    
    # print(f"response_words_re: {response_words_re}")
    # print(f"response_words: {response_words}")
    # print(len(response_words_re), len(response_words))
    if max(len(response_words_re), len(response_words)) < 2:
        response_words = "\n".join(response_words).split('\n')
        response_words = [word.strip().lower().strip('-') for word in response_words]
        
    elif len(response_words_re) > len(response_words):
        response_words = response_words_re
        response_words = [word.strip().lower().strip('-') for word in response_words]
        
    if response_text.count("\n") <= 3 and len(response_words_raw) < 4: 
        if not to_string:
            return [] 
        else:
            return ""
    
    
    
    # make it set to remove duplicates
    new_list = []
    for word in response_words:
        if chinese_flag:
            # print(f"word before parsing: {word}")
            parse_chinese = re.findall(r'[\u4e00-\u9fff]+', word)
            word = ''.join(parse_chinese)
        if word not in new_list and word != '':
            word = word.replace('  ', ' ')  # replace multiple spaces with a single space
            new_list.append(word)
        response_words = new_list
    if not to_string:
        
        return response_words
    else:
        return ", ".join(response_words)

In [3]:
with open('./swow_en_results.pkl', 'rb') as f:
    swow_en_results = pickle.load(f)
    
with open('./swow_zh_results.pkl', 'rb') as f:
    swow_zh_results = pickle.load(f)

In [5]:
swow_en_results.columns

Index(['Cue Word', 'Ground Truth Associated Words', 'Prompt Type',
       'Model Type', 'Generated Associated Words'],
      dtype='object')

In [151]:
swow_en_results_gaw = swow_en_results['Generated Associated Words']
swow_zh_results_gaw = swow_zh_results['Generated Associated Words']

## Test English Parsing

In [230]:
random_ind = random.randint(0, len(swow_en_results_gaw) - 1)

response_text = swow_en_results_gaw[random_ind]

print("Generated Associated Words:")
print(response_text)

response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

Generated Associated Words:
list of word associations for 'them':

* us, group, them, others, people, those, enemy, one, mice, folks, places, strangers, animals, vs, friends, over, unknown, awful, cough, pop  group, family, cave, it, evil, vs  us, group  of  people, song, feeling, ppl, everything, others  people, song  by  blondes, alike, target, browse, character, distance, neighbors, memories, person, song  by  cameron  decay, memories  of  past, male
response_words_re: []
response_words: ["list of word associations for 'them':\n\n* us", 'group', 'them', 'others', 'people', 'those', 'enemy', 'one', 'mice', 'folks', 'places', 'strangers', 'animals', 'vs', 'friends', 'over', 'unknown', 'awful', 'cough', 'pop  group', 'family', 'cave', 'it', 'evil', 'vs  us', 'group  of  people', 'song', 'feeling', 'ppl', 'everything', 'others  people', 'song  by  blondes', 'alike', 'target', 'browse', 'character', 'distance', 'neighbors', 'memories', 'person', 'song  by  cameron  decay', 'memories  of 

In [124]:
response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

doing additional regex parsing for response_text: Here are some words I associate with "inspire":

inspire
motivate
encourage
inspiration
motivation
church
strength
hero
life
big
sun
music
people
success
response_words_re: ['inspire\r', 'motivate\r', 'encourage\r', 'inspiration\r', 'motivation\r', 'church\r', 'strength\r', 'hero\r', 'life\r', 'big\r', 'sun\r', 'music\r', 'people\r']
response_words: ['here are some words i associate with "inspire":\r\n\r\ninspire\r\nmotivate\r\nencourage\r\ninspiration\r\nmotivation\r\nchurch\r\nstrength\r\nhero\r\nlife\r\nbig\r\nsun\r\nmusic\r\npeople\r\nsuccess']
13 1
Parsed Response List:
['inspire', 'motivate', 'encourage', 'inspiration', 'motivation', 'church', 'strength', 'hero', 'life', 'big', 'sun', 'music', 'people']


## Test Chinese Parsing

In [235]:
random_ind = random.randint(0, len(swow_zh_results_gaw) - 1)

response_text = swow_zh_results_gaw[random_ind]

print("Generated Associated Words:")
print(response_text)

response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

Generated Associated Words:
当然可以，以下是我对“晚上”的一些词语联想：

- 夜晚
- 夜空
- 晚安
- 白天
- 星空
- 黑白 Television
- 床
- 白开水
- 幕色实验
- 夜宵
- 睡眠
- 午饭
- 火锅
- 罗跌
- 绝星 citrader
- 早睡早起
- 路灯
- 安静
- 媳妇

Parsed Response List:
夜晚, 夜空, 晚安, 白天, 星空, 黑白, 床, 白开水, 幕色实验, 夜宵, 睡眠, 午饭, 火锅, 罗跌, 绝星, 早睡早起, 路灯, 安静, 媳妇


In [218]:
response_words = parse_response_text(response_text)

print("Parsed Response List:")
print(response_words)

response_text after removing prefix: 
长辈们唠叨的声音
喧闹的动物声响
鸡鸣声
叫卖声
喋喋不休的对话
乱七八糟的
売货声
杂乱声
鸡叫声
吵闹声
吉他声音


 * 生物学和动物方面（如动物
response_words_re: []
response_words: ['长辈们唠叨的声音\n喧闹的动物声响\n鸡鸣声\n叫卖声\n喋喋不休的对话\n乱七八糟的\n売货声\n杂乱声\n鸡叫声\n吵闹声\n吉他声音\n\n\n * 生物学和动物方面（如动物']
0 1
Parsed Response List:
['长辈们唠叨的声音', '喧闹的动物声响', '鸡鸣声', '叫卖声', '喋喋不休的对话', '乱七八糟的', '売货声', '杂乱声', '鸡叫声', '吵闹声', '吉他声音', '生物学和动物方面如动物']


In [236]:
# process swow_en_results_gaw
swow_en_results['Parsed Associated Words'] = swow_en_results_gaw.apply(parse_response_text)

# process swow_zh_results_gaw
swow_zh_results['Parsed Associated Words'] = swow_zh_results_gaw.apply(parse_response_text)

# Save the processed results
with open('./swow_en_results_processed.pkl', 'wb') as f:
    pickle.dump(swow_en_results, f)
with open('./swow_zh_results_processed.pkl', 'wb') as f:
    pickle.dump(swow_zh_results, f)

In [237]:
swow_zh_results

,Cue Word,Ground Truth Associated Words,Prompt Type,Model Type,Generated Associated Words,Parsed Associated Words
0,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Complex,Qwen/Qwen2.5-7B-Instruct,监狱\n囚犯\n监管\n安全\n囚禁\n守卫\n执法\n拘留\n看守\n惩罚\n修正\n警察...,"监狱, 囚犯, 监管, 安全, 囚禁, 守卫, 执法, 拘留, 看守, 惩罚, 修正, 警察..."
1,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Simple,Qwen/Qwen2.5-7B-Instruct,好的，以下是与“狱警”这个词汇相关的联想词列表：\n\n1. 监狱\n2. 犯人\n3. 约...,"监狱, 犯人, 约束, 权力, 走廊, 穿囚服的人, 监控摄像头, 手铐, 严正, 安全"
2,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Complex,sukai/qwen_swow_zh,警察， 灯塔， 犯罪， 枪， 灯泡， 手铐， 鞭子， 鞭打， 警戒， 砖头， 警棍， 电影，...,"警察, 灯塔, 犯罪, 枪, 灯泡, 手铐, 鞭子, 鞭打, 警戒, 砖头, 警棍, 电影,..."
3,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Simple,sukai/qwen_swow_zh,狱警：\n1. 警服\n2. 铁窗\n3. 犯罪\n4. 监狱风云\n5. 手铐\n6. 灯...,"警服, 铁窗, 犯罪, 监狱风云, 手铐, 灯泡, 负责任, 罪犯, 栅栏, 执勤"
4,狱警,监狱， 警察， 警棍， 囚犯， 制服， 罪犯， 犯人， 凶， 看守， 坐牢， 严厉， 警犬，...,Complex,sukai/qwen_ppo_zh,囚犯 监狱 看守 警戒 守卫 检查站 审讯 警察 手铐 警棍 对讲机 放风 禁闭 保安 安全...,"囚犯, 监狱, 看守, 警戒, 守卫, 检查站, 审讯, 警察, 手铐, 警棍, 对讲机, ..."
...,...,...,...,...,...,...
12043,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Simple,meta-llama/Meta-Llama-3.1-8B-Instruct,"下面是与 ""悔改"" 相关的立即联想词：\n\n1. 后悔\n2. 亡羊补牢\n3. 复仇\n...","后悔, 亡羊补牢, 复仇, 悔过, 道歉, 救赎, 反思, 青;color_scale群 控..."
12044,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Complex,sukai/llama_swow_zh,犯错， 后悔， 补偿， 手工， Clement， 杨阳Mine， 犯错， 改正， 得汇， 矫...,"犯错, 后悔, 补偿, 手工, 杨阳, 改正, 得汇, 矫行, 我, 两面剑客, 懊心, 好..."
12045,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Simple,sukai/llama_swow_zh,"我立马可以返回以下列表：\n""Yes, please""\n\n以下是联想词：\n1. 痛苦，...","痛苦, 书面语, 道歉, 改正, 个字, 回顾, 回想, 后悔, 彷徨, 纣, 后悔药, 烦..."
12046,悔改,后悔， 不知悔改， 错误， 不知， 犯错， 监狱， 犯人， 忏悔， 哭泣， 改过自新， 低头...,Complex,sukai/llama_ppo_zh,悔改 \n谦逊 \n退省 \n自责 \n反省 \n批判 \n清醒 \n醒悟 ...,悔改谦逊退省自责反省批判清醒醒悟忏悔矫正良心自我批评改过且安好悔彻付出努力责任自律重新开始上...
